# Generating Graph Dataset

In [ ]:
from random import shuffle
import itertools, pandas as pd
from typing import List, Sequence

class Node(object):
	def __init__(self, id):
		self.id = id
		self.children = []
		self.parents = []

	def __eq__(self, other):
		return self.id == other.id

	def __hash__(self):
		return hash(self.id)

	def __str__(self):
		return 'n(' + str(self.id) + ')'

	def __repr__(self):
		return 'n(' + str(self.id) + ')'


def generate_graph_with_lookahead(lookahead, num_paths):   # ← still unused
    """
    Build a graph of the form

           0
        /  |  \
       …  …  …            ← num_paths first‑layer nodes
       |  |  |
       …  …  …            ← (lookahead‑1) internal layers
        \  |  /
          7 7 7           ← num_paths distinct Node objs that all have id = sink_id

    Returns (vertices, start, end_nodes)
    """
    if lookahead < 1:
        raise ValueError("lookahead must be ≥ 1")
    if num_paths < 1:
        raise ValueError("num_paths must be ≥ 1")

    num_unique = 1 + num_paths * lookahead
    sink_id    = num_unique                      # e.g. 7 in the diagram

    vertices = [Node(i) for i in range(num_unique)]

    # add *num_paths* distinct Node objects that all share id = sink_id
    sink_nodes = []
    for _ in range(num_paths):
        sink_nodes.append(Node(sink_id))
    vertices.extend(sink_nodes)


    root = vertices[0]
    # 2‑a. root  → first‑layer nodes
    for p in range(num_paths):
        idx = 1 + p
        root.children.append(vertices[idx])
        vertices[idx].parents.append(root)

    # each path’s internal chain + edge to its sink
    offset = 1 + num_paths                   # where the second layer starts
    step   = lookahead - 1                   # nodes per path AFTER first layer

    for p in range(num_paths):
        prev = vertices[1 + p]               # first‑layer node for this path

        # internal nodes (if lookahead > 1)
        for d in range(step):                # d = 0 … step‑1
            idx = offset + p * step + d      # unique node for (p, d)
            curr = vertices[idx]
            prev.children.append(curr)
            curr.parents.append(prev)
            prev = curr

        # connect last internal node to its sink duplicate
        sink = sink_nodes[p]
        prev.children.append(sink)
        sink.parents.append(prev)

    sink_id = sink_nodes[0].id
    internal_nodes = [v for v in vertices if v.id != sink_id]

    new_ids_pool = list(range(sink_id))
    print(new_ids_pool)
    shuffle(new_ids_pool)

    src = 0
    for v in internal_nodes:
        # skip reserved indices and the sink_id itself
        while new_ids_pool[src] == sink_id:
            src += 1
        v.id = new_ids_pool[src]
        src += 1

    shuffle(vertices)

    return vertices, root, sink_nodes
def print_graph(vertices,  show_parents=False):
    """Print each node and its children (and optionally its parents)."""
    for v in sorted(vertices, key=lambda n: n.id):
        children_ids = [c.id for c in v.children]
        line = f"{v.id:>4}  ->  {children_ids}"
        if show_parents:
            parent_ids = [p.id for p in v.parents]
            line += f"    (parents: {parent_ids})"
        print(line)

def build_model_input(vertices, start, end, tokens):
    QUERY, EDGE, PATH = tokens
    prefix = []
    for v in vertices:
        for child in v.children:
            prefix.extend([EDGE, v.id, child.id])
    prefix.extend([QUERY, start.id, end.id, PATH, start.id])  # include start after P

    # convert numeric stream to human‑readable tokens
    human = []
    for tok in prefix:
        if tok == EDGE:
            human.append("E")
        elif tok == QUERY:
            human.append("Q")
        elif tok == PATH:
            human.append("P")
        else:
            human.append(str(tok))
    return "".join(human), prefix


def build_corrupted_input(adj_list, paths, sink_id,edge_token):
  inps = []
  dummy_id = sink_id + 1

  for keep_idx, keep_path in enumerate(paths):
      ablated_list = adj_list.copy()
      keep_set = set(keep_path)

      for i in range(len(adj_list) - 2):
          if adj_list[i] != edge_token:
              continue

          u, v = adj_list[i + 1], adj_list[i + 2]

          # If this edge belongs to a path we want to ablate (i.e., not the one we're keeping)
          # and both nodes are part of some path (to avoid touching unrelated edges)
          if (u not in keep_set or v not in keep_set):
              for j, other_path in enumerate(paths):
                  if j == keep_idx:
                      continue
                  if u in other_path and v in other_path:
                      ablated_list[i + 2] = dummy_id  # redirect destination

      inps.append(ablated_list)

  return inps

def convert_to_str(graph, tokens):
  human = []
  QUERY, EDGE, PATH = tokens
  for tok in graph:
      if tok == EDGE:
          human.append("E")
      elif tok == QUERY:
          human.append("Q")
      elif tok == PATH:
          human.append("P")
      else:
          human.append(str(tok))
  return "".join(human)

def find_all_sink_paths(root: Node, sink_id: int, max_paths: int = 3) -> List[List[int]]:
    """
    Depth‑first search that returns up to `max_paths` distinct
    root‑to‑sink paths, where a sink is any node whose `id == sink_id`.

    Each path is returned as a list of node ids, including the root and sink.
    """
    paths: List[List[int]] = []

    def dfs(node: Node, path: List[int]):
        # Early exit if we've already collected enough
        if len(paths) >= max_paths:
            return

        # Found a sink → store the path
        if node.id == sink_id:
            paths.append(path.copy())
            return

        # Explore children
        for child in node.children:
            # Avoid cycles (graph is a DAG, but be safe)
            if child.id not in path:
                dfs(child, path + [child.id])

    dfs(root, [root.id])
    return paths

# parameters
n_samples = 2
lookahead = 2
num_paths = 3
max_input_size = 256

def token_constants(max_input_size):
    QUERY_PREFIX_TOKEN  = (max_input_size - 5) // 3 + 4
    EDGE_PREFIX_TOKEN   = (max_input_size - 5) // 3 + 2
    PATH_PREFIX_TOKEN   = (max_input_size - 5) // 3 + 1
    return QUERY_PREFIX_TOKEN,EDGE_PREFIX_TOKEN, PATH_PREFIX_TOKEN

tokens = token_constants(max_input_size)

records = []
for _ in range(n_samples):
    verts, start, sinks = generate_graph_with_lookahead(lookahead, num_paths)
    # choose one sink for the query (e.g. first)
    end = sinks[0]
    inp, adj_list = build_model_input(verts, start, end, tokens)
    paths = find_all_sink_paths(start,end.id)
    exp_output1 = "".join(str(x) for x in paths[0])
    exp_output2 = "".join(str(x) for x in paths[1])
    exp_output3 = "".join(str(x) for x in paths[2])
    corrupted_inps = build_corrupted_input(adj_list,paths,end.id,tokens[1])
    corrupt_inp1 = convert_to_str(corrupted_inps[0],tokens)
    corrupt_inp2 = convert_to_str(corrupted_inps[1],tokens)
    corrupt_inp3 = convert_to_str(corrupted_inps[2],tokens)
    records.append({
    "model_input": inp,
    "output1": exp_output1,
    "output2": exp_output2,
    "output3": exp_output3,
    "corrupt_inp1": corrupt_inp1,
    "corrupt_inp2": corrupt_inp2,
    "corrupt_inp3": corrupt_inp3
})

df = pd.DataFrame(records)


In [ ]:
print(df.head)

# Path Patching


In [ ]:
import google.colab
!pip install transformer_lens
!pip install torch torchvision torchaudio torchtext
!pip install circuitsvis
!pip install accelerate
!pip install tiktoken


In [ ]:
# Import stuff
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import einops
from fancy_einsum import einsum
import tqdm.notebook as tqdm
import random
from pathlib import Path
import plotly.express as px
from torch.utils.data import DataLoader

from jaxtyping import Float, Int
from typing import List, Union, Optional
from functools import partial
import copy
import plotly.io as pio
import accelerate

import itertools
from transformers import AutoModelForCausalLM, AutoConfig, AutoTokenizer
import dataclasses
import datasets
from IPython.display import HTML
import circuitsvis as cv


import transformer_lens
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookedRootModule,
    HookPoint,
)  # Hooking utilities
from transformer_lens import HookedTransformer, HookedTransformerConfig, FactoredMatrix, ActivationCache

In [ ]:
torch.set_grad_enabled(False)

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu' #set the device if a GPU is available

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

model = HookedTransformer.from_pretrained(
    "qwen-1.8b-chat",
    center_unembed=True,
    fold_ln=True,
    low_cpu_mem_usage=True
).half()

model.cfg.dtype = torch.float16

In [ ]:

prompt_template = """ You are given a directed acyclic graph encoded as a sequence of tokens.
Each edge is represented by:
  Euv meaning there is a directed edge from node u to node v
A query is represented by:
  Qst   → meaning: is there a path from node s to node t?
Below are some examples:
{examples}
Here is the input:
{model_input}
Answer:P{answer}"""


examples = """
Input:E15E57E67E02E03E01E34E26E47Q07P0
Answer:P0257
Input:E17E31E47E25E20E23E06E67E54Q27P2
Answer:P2547
Input:E07E36E13E12E14E40E67E57E25Q17P1
Answer:P1367
""".strip()


chatml_header = (
    "<|im_start|>system\n"
    "You are a graph reasoning assistant. Given edges and a query, "
    "return a valid path using the format P s ... t.<|im_end|>\n"
    "<|im_start|>user\n"
)
chatml_footer = "<|im_end|>\n<|im_start|>assistant\n"

# --- 2. Build prompts for every row in the DataFrame -----------------------
def make_chatml_prompt(model_input: str,answer) -> str:
    """Fill user prompt, then wrap in ChatML blocks."""
    user_prompt = prompt_template.format(
        examples=examples,
        model_input=model_input.strip(),
        answer = answer.strip()
    )
    return f"{chatml_header}{user_prompt}{chatml_footer}"

chatml_prompt = make_chatml_prompt("E16E67E57E05E32E41E43E40E27Q47P4","4167")
print(chatml_prompt)
# utils.test_prompt(chatml_prompt,answer='P4167', model=model, prepend_space_to_answer=False)

In [ ]:
"""
- ablations + looking at attention map to see if it stores multiple paths, use path patch to find which attention heads are involved
- apply the pairs mechn interp to find the algorithm more clearly if it exists- in this multi-path example
""""




In [ ]:
import torch as t
import tqdm
from functools import partial
from typing import Callable, Optional, Tuple
from jaxtyping import Float
from transformer_lens import HookedTransformer, ActivationCache

n_heads = model.cfg.n_heads
n_layers = model.cfg.n_layers
def patch_or_freeze_head_vectors(
	orig_head_vector: Float[t.Tensor, "batch pos head_index d_head"],
	hook,
	new_cache: ActivationCache,
	orig_cache: ActivationCache,
	head_to_patch: Tuple[int, int],
) -> Float[t.Tensor, "batch pos head_index d_head"]:
    '''
    Replaces the output of a specific attention head (head_to_patch) with its value from new_cache,
    while freezing all other heads' outputs to their original values from orig_cache ( leave it unchanged in this case for a more space optimal solution)
    '''
    cur_layer = hook.layer()
    orig_cache_comp = orig_cache[hook.name]
    new_cache_comp = new_cache[hook.name]

    if head_to_patch[0] == cur_layer:
      orig_head_vector[:,:,head_to_patch[1],:] = new_cache_comp[:,:,head_to_patch[1],:].to(orig_head_vector.device)


    return orig_head_vector


def get_path_patch_head_to_final_resid_post(
    model: HookedTransformer,
    patching_metric: Callable,
    new_dataset,
    answers,
    orig_dataset,
    all_answers
) -> Float[t.Tensor, "layer head"]:


    ## run model on clean and corrupted inputs, cache the head outputs
    model.reset_hooks()
    z_name_filter = lambda name: name.endswith("z")
    results = t.zeros(model.cfg.n_layers, model.cfg.n_heads, device=device, dtype=t.float32)
    results_lse  = torch.zeros_like(results)

    ## run forward paths
    clean_logits, orig_cache = model.run_with_cache(orig_dataset,names_filter = z_name_filter,device= 'cpu')
    _ , new_cache = model.run_with_cache(new_dataset,names_filter = z_name_filter, device = 'cpu')


## run model on clean input with sender head patched from the corrupted input, freeze other heads, cache final value of residual stream
    for sender_layer in tqdm.tqdm(list(range(n_layers))):
      for sender_head in range(n_heads):
        hook_fn = partial(patch_or_freeze_head_vectors,
                          orig_cache = orig_cache,
                          new_cache=new_cache,
                          head_to_patch=(sender_layer,sender_head))
        with t.no_grad():
          patched_logits = model.run_with_hooks(
                orig_dataset,
                fwd_hooks = [(utils.get_act_name('z', sender_layer,'attn'), hook_fn)],
                return_type='logits'
            )
        ans_pos = 263
        delta = patching_metric(clean_logits,answers,ans_pos) - patching_metric(patched_logits,answers,ans_pos)
        clean_lse  = lse_correct_path_mass(clean_logits,  all_answers, ans_pos)
        patch_lse  = lse_correct_path_mass(patched_logits, all_answers, ans_pos)
        delta_lse  = (clean_lse - patch_lse).mean()
        results[sender_layer, sender_head] = delta.mean().cpu()
        results_lse[sender_layer, sender_head] = delta_lse.cpu()
        del patched_logits
        t.cuda.empty_cache()
    return results, results_lse


In [ ]:

def mean_answer_logit(
    logits: torch.Tensor,          # (batch, seq_len, vocab)
    answer_tokens: torch.Tensor,      # (batch, path_length)
    p_pos: int                     # sequence index where 'P' itself appears
) -> torch.Tensor:
    """
    Compute the mean next‑token logit for the *node tokens* of the answer,
    for every prompt in the batch.

    Parameters
    ----------
    logits : torch.Tensor
        Model logits of shape (B, seq_len, vocab).
    answer_tokens : list[int]
        Token IDs for the path nodes *after* 'P', e.g. [4,1,6,7].
    p_pos : int
        The first node token is assumed to be at position p_pos + 1.

    Returns
    -------
    torch.Tensor
        Shape (B,) – average logit per batch item.
    """
    batch, _, _ = logits.shape
    print("shape",logits.shape)
    out = torch.empty(batch, dtype=logits.dtype, device=logits.device)

    # ---- loop over batch ----
    for b in range(batch):
        total = 0.0  # scalar tensor
        batch_answer = answer_tokens[b]
        n_nodes = len(batch_answer)

        # ---- loop over node tokens (skip the 'P') ----
        for offset in range(n_nodes):
            node_tok_id = int(batch_answer[offset])
            seq_pos  = p_pos + offset + 1
            token_logit = logits[b, seq_pos, node_tok_id]
            total += token_logit.item()
        out[b] = total / n_nodes
    return out

def lse_correct_path_mass(
    logits: torch.Tensor,              # (batch, seq, vocab)
    all_answer_tokens: torch.Tensor,   # (batch, num_paths, path_len) ints
    p_pos: int
) -> torch.Tensor:                     # (batch,) fp16
    """
    Compute log‑sum‑exp of summed logits over *all* valid paths.
    """
    B, P, L = all_answer_tokens.shape
    device  = logits.device
    out     = torch.empty(B, dtype=logits.dtype, device=device)

    for b in range(B):
        path_sums = []
        for p in range(P):
            total = 0.0
            for offset in range(L):
                tok_id  = int(all_answer_tokens[b, p, offset])
                seq_pos = p_pos + 1 + offset      # skip 'P'
                total  += logits[b, seq_pos, tok_id].item()
            path_sums.append(total)
        out[b] = torch.logsumexp(torch.tensor(path_sums, dtype=logits.dtype, device=device), dim=0)
    return out

In [ ]:
## preparing data for path patching experiments

df["clean_input_prompt1"] = df.apply(
    lambda row: make_chatml_prompt(row["model_input"], row["output1"]),
    axis=1
)
df["clean_input_prompt2"] = df.apply(
    lambda row: make_chatml_prompt(row["model_input"], row["output2"]),
    axis=1
)
df["clean_input_prompt3"] = df.apply(
    lambda row: make_chatml_prompt(row["model_input"], row["output3"]),
    axis=1
)
df["corrupt_prompt_1"] = df.apply(
    lambda row: make_chatml_prompt(row["corrupt_inp1"], row["output1"]),
    axis=1
)

df["corrupt_prompt_2"] = df.apply(
    lambda row: make_chatml_prompt(row["corrupt_inp2"], row["output2"]),
    axis=1
)

df["corrupt_prompt_3"] = df.apply(
    lambda row: make_chatml_prompt(row["corrupt_inp3"], row["output3"]),
    axis=1
)
df["clean_input_tokens1"] = df["clean_input_prompt1"].apply(model.to_tokens)
df["clean_input_tokens2"] = df["clean_input_prompt2"].apply(model.to_tokens)
df["clean_input_tokens3"] = df["clean_input_prompt3"].apply(model.to_tokens)

df["corrupt_tokens_1"] = df["corrupt_prompt_1"].apply(model.to_tokens)
df["corrupt_tokens_2"] = df["corrupt_prompt_2"].apply(model.to_tokens)
df["corrupt_tokens_3"] = df["corrupt_prompt_3"].apply(model.to_tokens)
df["output_tokens_1"] = df["output1"].apply(model.to_tokens)
df["output_tokens_2"] = df["output2"].apply(model.to_tokens)
df["output_tokens_3"] = df["output3"].apply(model.to_tokens)


In [ ]:
from pprint import pprint

def show_tokens(prompt: str, model):
    """Pretty‑print (index, repr(str_token), token_id)."""
    str_toks = model.to_str_tokens(prompt)
    ids      = model.to_tokens(prompt).squeeze(0)

    table = [(i, repr(tok), int(tid)) for i, (tok, tid) in enumerate(zip(str_toks, ids))]
    pprint(table, compact=True, width=120)

# --- call it ---
show_tokens(chatml_prompt, model)

In [ ]:
import torch

def stack_answer_paths(df, cols):
    """
    Parameters
    ----------
    df   : pandas.DataFrame
    cols : list[str]
        Column names in order (e.g. ["output_tokens_1", "output_tokens_2", "output_tokens_3"])
    Returns
    -------
    torch.Tensor
        Shape (batch, num_paths, path_len)
    """
    batch_tensors = []

    for _, row in df.iterrows():
        path_tensors = []
        for c in cols:
            tok = row[c]
            path_tensors.append(tok.squeeze())

        # (= num_paths, path_len)
        stacked_paths = torch.stack(path_tensors)    # same length asserted
        batch_tensors.append(stacked_paths)

    # (batch, num_paths, path_len)
    return torch.stack(batch_tensors)


# --- usage ------------------------------------------------------------
cols = ["output_tokens_1", "output_tokens_2", "output_tokens_3"]
all_answers = stack_answer_paths(df, cols)



In [ ]:
## running path patching
def stack_token_column(series):
    return t.stack([tok.squeeze(0) for tok in series]).to(device)

clean_batch1  = stack_token_column(df["clean_input_tokens1"])
clean_batch2  = stack_token_column(df["clean_input_tokens2"])
clean_batch3  = stack_token_column(df["clean_input_tokens3"])


corrupt_batch_1  = stack_token_column(df["corrupt_tokens_1"])
corrupt_batch_2  = stack_token_column(df["corrupt_tokens_2"])
corrupt_batch_3  = stack_token_column(df["corrupt_tokens_3"])
answer_nodes1 = t.stack([tok.squeeze(0) for tok in df["output_tokens_3"]])

results, lse_results = get_path_patch_head_to_final_resid_post(model,mean_answer_logit,corrupt_batch_1,answer_nodes1,clean_batch1,all_answers)



In [ ]:
px.imshow(results.cpu(), labels={'x':'Head', 'y':"Layer"})